[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabin2004/Machine-Learning-Bootcamp/blob/main/Module_08_Neural_Networks/02_keras_basics.ipynb)

# Episode 21 – Building Neural Networks with Keras

**Machine Learning Bootcamp** | Module 08

---

## 🎯 Learning Objectives
- Build, compile, and train neural networks with Keras
- Add regularisation (Dropout, BatchNorm) to prevent overfitting
- Visualise training history and evaluate on test data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

print(f'TensorFlow: {tf.__version__}  |  Keras: {keras.__version__}')
sns.set_theme(style='whitegrid')

## 1. Prepare Data

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 2. Build the Model

In [ ]:
def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(X_train.shape[1])
model.summary()

## 3. Train the Model

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=20, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)
print(f'Stopped at epoch {len(history.history["loss"])}')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(history.history[metric],        label='Train')
    ax.plot(history.history[f'val_{metric}'], label='Val')
    ax.set_xlabel('Epoch'); ax.set_ylabel(metric.capitalize())
    ax.set_title(f'Training {metric.capitalize()}')
    ax.legend()

plt.tight_layout(); plt.show()

## 4. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
print()
print(classification_report(y_test, y_pred, target_names=data.target_names))

## 🏋️ Exercises

1. Remove `BatchNormalization` and `Dropout`. Does the model overfit more?
2. Add a third hidden layer. Does it improve performance?
3. Try the `RMSprop` and `SGD` optimisers. Compare convergence speed.
4. Save the trained model (`model.save(...)`) and reload it with `keras.models.load_model(...)`.

---
**Next ▶ [Module 09 – Hyperparameter Tuning & Pipelines](../Module_09_Hyperparameter_Tuning/01_hyperparameter_tuning.ipynb)**